# TP4: Distillation de Modèles de Raisonnement (DASD)

## 🎯 Objective
Implement **Distribution-Aligned Sequence Distillation (DASD)** to transfer reasoning capabilities from a large teacher model to a smaller student model.

## 📋 Workflow Overview

### Day 1: Setup & Training
1. **Phase 1-2**: Install environment & explore reference dataset
2. **Phase 3**: Generate dataset via API (teacher model)
3. **Phase 4**: Apply Divergence-Aware Sampling (DAS) filtering
4. **Phase 5**: Train Stage 1 (low temp) & Stage 2 (high temp)

### Day 2: Evaluation
5. **Phase 7**: Verify training results
6. **Phase 8**: Test model quality
7. **Phase 9**: Quantitative benchmark evaluation
8. **Phase 10**: Compile report

## ⚠️ Important Notes
- Replace `YOUR_API_KEY_HERE` with your Infomaniak API key
- This notebook is designed for **Kaggle with GPU** (2x T4 GPUs available)
- Enable GPU: Settings → Accelerator → GPU T4 x2
- Checkpoints saved to `/kaggle/working/` (persistent storage)
- Adjust sample sizes based on API limits and time constraints

---

## 🎮 Kaggle Setup Tips

Before starting, configure your Kaggle notebook:

1. **Enable GPU**: Click ⚙️ Settings (right panel) → Accelerator → Select "GPU T4 x2"  
2. **Enable Internet**: Settings → Internet → **ON** (required for API calls and pip installs)
3. **Persistence**: `/kaggle/working/` persists between sessions - save important files there
4. **Session Time**: Free Kaggle sessions last up to 12 hours
5. **Verify GPU**: Run the "Check GPU environment" cell to confirm GPU is active

---

# Finetune Qwen3 with LLaMA Factory

Please use **Kaggle with GPU** (2x T4 GPUs available for free)!

Project homepage: https://github.com/hiyouga/LLaMA-Factory

## Install Dependencies

In [ ]:
%cd /kaggle/working/
%rm -rf LLaMA-Factory
%git clone --depth 1 https://github.com/hiyouga/LLaMA-Factory.git
%cd LLaMA-Factory
%ls
%pip install -e .[torch,bitsandbytes]
%pip install -U bitsandbytes>=0.46.1

/content
Cloning into 'LLaMA-Factory'...
remote: Enumerating objects: 614, done.
remote: Counting objects: 100% (614/614), done.
remote: Compressing objects: 100% (455/455), done.
remote: Total 614 (delta 150), reused 383 (delta 101), pack-reused 0 (from 0)
Receiving objects: 100% (614/614), 5.24 MiB | 27.65 MiB/s, done.
Resolving deltas: 100% (150/150), done.
/content/LLaMA-Factory
assets/       docker/    LICENSE      pyproject.toml  requirements/  tests/
CITATION.cff  docs/      Makefile     README.md       scripts/       tests_v1/
data/         examples/  MANIFEST.in  README_zh.md    src/
Obtaining file:///content/LLaMA-Factory
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Installing backend dependencies ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for llamafactory (pyproject.toml) ... done
  Created wheel for llamafactory: filename=lla

In [ ]:
# Create output directories in Kaggle's persistent storage
!mkdir -p /kaggle/working/tp4_dasd/outputs
!mkdir -p /kaggle/working/tp4_dasd/data
import os
print("✅ Output directories created")
print(f"📁 Working directory: {os.getcwd()}")

## Setup Kaggle Working Directory

### Check GPU environment (Kaggle)

In [ ]:
import torch
try:
  assert torch.cuda.is_available() is True
  print(f"✅ GPU Available: {torch.cuda.get_device_name(0)}")
  print(f"📊 GPU Count: {torch.cuda.device_count()}")
except AssertionError:
  print("❌ No GPU detected! Enable GPU in Kaggle: Settings → Accelerator → GPU T4 x2")
print("cuda OK")

cuda OK


## Phase 2: Study the Reference Dataset

Before generating our own data, let's explore the official DASD dataset to understand:
- Data format (instruction/response structure)
- Use of reasoning tags (`<reasoning>` or `<think>`)
- Response quality and length patterns

In [ ]:
from datasets import load_dataset
import json

# Load the official DASD reference dataset
print("Loading reference dataset...")
reference_dataset = load_dataset(
    "Alibaba-Apsara/Superior-Reasoning-SFT-gpt-oss-120b",
    "stage1",
    split="train"
)

print(f"\n📊 Dataset size: {len(reference_dataset)} examples")
print(f"📋 Features: {reference_dataset.features}")

# Examine a few examples
print("\n" + "="*80)
print("EXAMPLE 1:")
print("="*80)
example = reference_dataset[0]
print(json.dumps(example, indent=2, ensure_ascii=False)[:1000] + "...")

# Analyze response lengths
response_lengths = []
has_reasoning_tags = 0

for example in reference_dataset.select(range(min(100, len(reference_dataset)))):
    # Adjust field names based on actual structure
    if 'conversations' in example:
        for conv in example['conversations']:
            if conv.get('from') == 'assistant' or conv.get('role') == 'assistant':
                content = conv.get('value', conv.get('content', ''))
                response_lengths.append(len(content))
                if '<reasoning>' in content or '<think>' in content:
                    has_reasoning_tags += 1

print(f"\n📈 Statistics (first 100 examples):")
if response_lengths:
    print(f"   - Avg response length: {sum(response_lengths)/len(response_lengths):.0f} chars")
    print(f"   - Max response length: {max(response_lengths)} chars")
    print(f"   - Responses with reasoning tags: {has_reasoning_tags}/{len(response_lengths)}")

## Phase 3: Generate Your Own Dataset via API

We'll use Infomaniak's API (OpenAI compatible) to generate responses from a teacher model.

### Step 3.1: Configure API and Choose Source Instructions

### Step 3.2: Generate Responses with Temperature Scheduling

Generate responses at two temperature levels:
- **Stage 1**: Low temperature (τ=0.3) for stable, high-quality responses
- **Stage 2**: High temperature (τ=0.9) for diverse reasoning paths

In [ ]:
from openai import OpenAI
import os
from datasets import load_dataset

# Configure Infomaniak API
INFOMANIAK_API_KEY = "YOUR_API_KEY_HERE"  # ⚠️ Replace with your API key
INFOMANIAK_BASE_URL = "https://api.infomaniak.com/2/ai/48/openai/v1"
TEACHER_MODEL = "qwen3"  # Choose from available models

client = OpenAI(
    api_key=INFOMANIAK_API_KEY,
    base_url=INFOMANIAK_BASE_URL
)

# Test API connection
print("🔍 Testing API connection...")
try:
    response = client.chat.completions.create(
        model=TEACHER_MODEL,
        messages=[{"role": "user", "content": "Say 'API OK' if you can read this."}],
        max_tokens=50
    )
    print(f"✅ API Response: {response.choices[0].message.content}")
except Exception as e:
    print(f"❌ API Error: {e}")

# Load source instructions (choose one dataset or create your own)
print("\n📚 Loading source instructions...")
# Option 1: GSM8K (math reasoning)
# source_dataset = load_dataset("gsm8k", "main", split="train")
# instructions = [ex["question"] for ex in source_dataset.select(range(200))]

# Option 2: Alpaca (general instructions)
source_dataset = load_dataset("tatsu-lab/alpaca", split="train")
instructions = [ex["instruction"] for ex in source_dataset.select(range(200)) if ex["instruction"].strip()]

print(f"✅ Loaded {len(instructions)} instructions")
print(f"\n📝 Example instruction: {instructions[0]}")

In [ ]:
import time
import json
from tqdm import tqdm

def generate_teacher_responses(instructions, temperature, max_samples=100):
    """
    Generate responses from teacher model with logprobs for DAS.
    
    Args:
        instructions: List of instruction strings
        temperature: Sampling temperature (0.3 for stage1, 0.9 for stage2)
        max_samples: Maximum number of samples to generate
    
    Returns:
        List of dicts with instruction, response, and logprobs
    """
    system_prompt = """You are a helpful assistant that reasons step by step. 
Always structure your reasoning inside <reasoning>...</reasoning> tags before giving your final answer. 
Be thorough in your reasoning process."""
    
    results = []
    
    for instruction in tqdm(instructions[:max_samples], desc=f"Generating (τ={temperature})"):
        try:
            response = client.chat.completions.create(
                model=TEACHER_MODEL,
                messages=[
                    {"role": "system", "content": system_prompt},
                    {"role": "user", "content": instruction}
                ],
                temperature=temperature,
                max_tokens=2000,
                logprobs=True,
                top_logprobs=1
            )
            
            content = response.choices[0].message.content
            logprobs = response.choices[0].logprobs.content if response.choices[0].logprobs else None
            
            results.append({
                "instruction": instruction,
                "response": content,
                "logprobs": [
                    {"token": lp.token, "logprob": lp.logprob} 
                    for lp in logprobs
                ] if logprobs else None,
                "temperature": temperature
            })
            
            # Rate limiting: adjust delay as needed
            time.sleep(0.5)
            
        except Exception as e:
            print(f"\n⚠️ Error on instruction '{instruction[:50]}...': {e}")
            continue
    
    return results

# Generate Stage 1 data (low temperature)
print("🎯 Generating Stage 1 dataset (τ=0.3)...")
stage1_data = generate_teacher_responses(instructions, temperature=0.3, max_samples=50)
print(f"✅ Generated {len(stage1_data)} Stage 1 examples")

# Save Stage 1
with open("stage1_raw.json", "w", encoding="utf-8") as f:
    json.dump(stage1_data, f, indent=2, ensure_ascii=False)

print(f"\n📁 Saved to stage1_raw.json")
print(f"📝 Example response:\n{stage1_data[0]['response'][:300]}...")

In [ ]:
def calculate_student_logprobs(instruction, response, tokenizer, model):
    """
    Calculate student model's log probabilities for a teacher response.
    
    Returns:
        List of (token, logprob) tuples for the response tokens
    """
    # Format the full conversation
    messages = [{"role": "user", "content": instruction}]
    prompt_str = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )
    
    full_text = prompt_str + response
    
    # Tokenize
    inputs = tokenizer(full_text, return_tensors="pt").to(model.device)
    prompt_len = len(tokenizer(prompt_str, add_special_tokens=False)["input_ids"])
    
    # Forward pass
    with torch.no_grad():
        outputs = model(**inputs)
        logits = outputs.logits[0, :-1, :]
        labels = inputs["input_ids"][0, 1:]
    
    # Calculate log probabilities
    loss_fct = torch.nn.CrossEntropyLoss(reduction='none')
    token_losses = loss_fct(logits, labels)
    token_logprobs = -token_losses.cpu().numpy()
    
    # Extract response portion
    response_logprobs = token_logprobs[prompt_len - 1:]
    response_token_ids = labels[prompt_len - 1:].cpu().numpy()
    
    return list(zip(response_token_ids, response_logprobs))


def calculate_sentence_divergence(instruction, response, teacher_logprobs, tokenizer, model):
    """
    Calculate divergence for each sentence in the response.
    
    Returns:
        List of dicts with sentence, p_teacher, p_student, divergence
    """
    # Get student logprobs
    student_token_logprobs = calculate_student_logprobs(instruction, response, tokenizer, model)
    
    # Split response into sentences
    sentences = nltk.tokenize.sent_tokenize(response)
    
    results = []
    teacher_cursor = 0
    student_cursor = 0
    
    for sent in sentences:
        # Collect teacher logprobs for this sentence
        sent_teacher_logprobs = []
        accumulated_text = ""
        
        while teacher_cursor < len(teacher_logprobs):
            token_data = teacher_logprobs[teacher_cursor]
            sent_teacher_logprobs.append(token_data["logprob"])
            accumulated_text += token_data["token"]
            teacher_cursor += 1
            
            if len(accumulated_text) >= len(sent):
                break
        
        # Calculate average teacher probability (geometric mean via log space)
        p_teacher = np.exp(np.mean(sent_teacher_logprobs)) if sent_teacher_logprobs else 0.0
        
        # Collect student logprobs for this sentence
        sent_student_logprobs = []
        accumulated_student = ""
        
        while student_cursor < len(student_token_logprobs):
            token_id, logprob = student_token_logprobs[student_cursor]
            sent_student_logprobs.append(logprob)
            token_str = tokenizer.decode([token_id])
            accumulated_student += token_str
            student_cursor += 1
            
            if len(accumulated_student) >= len(sent):
                break
        
        p_student = np.exp(np.mean(sent_student_logprobs)) if sent_student_logprobs else 0.0
        
        divergence = p_teacher - p_student
        
        results.append({
            "sentence": sent,
            "p_teacher": float(p_teacher),
            "p_student": float(p_student),
            "divergence": float(divergence)
        })
    
    return results


def classify_example_quality(sentence_stats, threshold_teacher=0.6, threshold_divergence=0.2):
    """
    Classify an example based on its sentence-level divergence.
    
    Returns:
        score: Quality score (higher is better)
        keep: Boolean whether to keep this example
        stats: Classification statistics
    """
    teacher_sentences = 0
    shared_sentences = 0
    student_sentences = 0
    
    for stat in sentence_stats:
        p_t = stat["p_teacher"]
        divergence = stat["divergence"]
        
        # Teacher Sentence: Teacher confident, student not
        if p_t > threshold_teacher and divergence > threshold_divergence:
            teacher_sentences += 1
        # Student Sentence: Student over-confident or teacher uncertain
        elif divergence < -threshold_divergence:
            student_sentences += 1
        # Shared: Both similar
        else:
            shared_sentences += 1
    
    total = len(sentence_stats)
    teacher_ratio = teacher_sentences / total if total > 0 else 0
    
    # Keep if sufficient "teacher sentences" and low noise
    keep = (teacher_ratio > 0.3) and (student_sentences < total * 0.3)
    score = teacher_ratio - (student_sentences / total if total > 0 else 0)
    
    return {
        "score": score,
        "keep": keep,
        "teacher_sentences": teacher_sentences,
        "shared_sentences": shared_sentences,
        "student_sentences": student_sentences,
        "total_sentences": total,
        "teacher_ratio": teacher_ratio
    }

print("✅ DAS functions defined")

In [ ]:
import torch
import numpy as np
import nltk
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

# Download sentence tokenizer
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)

# Configure and load student model (Qwen2.5-7B)
print("🔧 Loading student model for DAS...")
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
)

student_model_id = "unsloth/Qwen2.5-7B-Instruct-bnb-4bit"
student_tokenizer = AutoTokenizer.from_pretrained(student_model_id, trust_remote_code=True)
student_model = AutoModelForCausalLM.from_pretrained(
    student_model_id,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True
)
student_model.eval()

print(f"✅ Student model loaded: {student_model_id}")

## Phase 4: Divergence-Aware Sampling (DAS) Implementation

DAS filters training examples by analyzing sentence-level divergence between teacher and student models.

**Key Concept**: Keep examples where the teacher is confident (high P_T) but the student is uncertain (low P_S) → "Teacher Sentences"

In [ ]:
# Load previously generated data
with open("stage1_raw.json", "r", encoding="utf-8") as f:
    stage1_raw = json.load(f)

# Apply DAS filtering
print("🔍 Applying DAS filtering to Stage 1 data...")
filtered_stage1 = []
das_scores = []

for example in tqdm(stage1_raw, desc="DAS Filtering"):
    if not example.get("logprobs"):
        continue
    
    try:
        # Calculate sentence-level divergence
        sent_stats = calculate_sentence_divergence(
            example["instruction"],
            example["response"],
            example["logprobs"],
            student_tokenizer,
            student_model
        )
        
        # Classify quality
        quality = classify_example_quality(sent_stats)
        das_scores.append(quality["score"])
        
        if quality["keep"]:
            filtered_stage1.append({
                "instruction": example["instruction"],
                "response": example["response"],
                "das_score": quality["score"],
                "das_stats": quality
            })
    
    except Exception as e:
        print(f"\n⚠️ Error processing example: {e}")
        continue

print(f"\n📊 DAS Filtering Results:")
print(f"   - Original examples: {len(stage1_raw)}")
print(f"   - Kept after DAS: {len(filtered_stage1)}")
print(f"   - Filter rate: {len(filtered_stage1)/len(stage1_raw)*100:.1f}%")
print(f"   - Avg DAS score: {np.mean([ex['das_score'] for ex in filtered_stage1]):.3f}")

# Visualize score distribution
import matplotlib.pyplot as plt

plt.figure(figsize=(10, 4))
plt.hist(das_scores, bins=30, alpha=0.7, edgecolor='black')
plt.axvline(0, color='red', linestyle='--', label='Decision boundary')
plt.xlabel('DAS Score')
plt.ylabel('Frequency')
plt.title('DAS Score Distribution')
plt.legend()
plt.grid(alpha=0.3)
plt.show()

# Save filtered data
with open("stage1_filtered.json", "w", encoding="utf-8") as f:
    json.dump(filtered_stage1, f, indent=2, ensure_ascii=False)

print("\n✅ Filtered data saved to stage1_filtered.json")

### Apply DAS Filtering to Stage 1 Data

## Phase 5: Two-Stage Training

Train the student model in two stages:
- **Stage 1**: Low temperature data (τ=0.3) for stable reasoning
- **Stage 2**: High temperature data (τ=0.9) for diverse reasoning paths

### Convert to LLaMA Factory Format (ShareGPT)

Convert our filtered data to the ShareGPT format expected by LLaMA Factory.

### Train Stage 1 Model (Low Temperature)

In [ ]:
def convert_to_sharegpt_format(examples):
    """
    Convert examples to ShareGPT format for LLaMA Factory.
    
    ShareGPT format:
    {
        "conversations": [
            {"from": "user", "value": "instruction"},
            {"from": "assistant", "value": "response"}
        ]
    }
    """
    sharegpt_data = []
    
    for ex in examples:
        sharegpt_data.append({
            "conversations": [
                {"from": "user", "value": ex["instruction"]},
                {"from": "assistant", "value": ex["response"]}
            ]
        })
    
    return sharegpt_data

# Convert Stage 1 data
stage1_sharegpt = convert_to_sharegpt_format(filtered_stage1)

# Save in LLaMA Factory data directory
%mkdir -p /kaggle/working/LLaMA-Factory/data
with open("/kaggle/working/LLaMA-Factory/data/dasd_stage1.json", "w", encoding="utf-8") as f:
    json.dump(stage1_sharegpt, f, indent=2, ensure_ascii=False)

# Register dataset in dataset_info.json
dataset_info_path = "/kaggle/working/LLaMA-Factory/data/dataset_info.json"

with open(dataset_info_path, "r", encoding="utf-8") as f:
    dataset_info = json.load(f)

dataset_info["dasd_stage1"] = {
    "file_name": "dasd_stage1.json",
    "formatting": "sharegpt",
    "columns": {
        "messages": "conversations"
    }
}

with open(dataset_info_path, "w", encoding="utf-8") as f:
    json.dump(dataset_info, f, indent=2, ensure_ascii=False)

print(f"✅ Converted {len(stage1_sharegpt)} examples to ShareGPT format")
print(f"✅ Registered 'dasd_stage1' dataset in LLaMA Factory")
print(f"\n📋 Example in ShareGPT format:")
print(json.dumps(stage1_sharegpt[0], indent=2, ensure_ascii=False)[:500] + "...")

---
## 📝 Report Checklist

For your final report (PDF, 4-6 pages), include:

### 1. Introduction
- [ ] Problem statement: Why distill large models?
- [ ] DASD approach overview
- [ ] Your methodology choices

### 2. Methodology
- [ ] Teacher model and API configuration
- [ ] Source instruction dataset choice
- [ ] DAS implementation details
- [ ] Training configuration (Stage 1 & 2)

### 3. Results
- [ ] Dataset statistics (before/after DAS)
- [ ] DAS score distribution plot
- [ ] Training loss curves
- [ ] Example model outputs
- [ ] Quantitative evaluation results

### 4. Discussion
- [ ] Analysis of improvement (or lack thereof)
- [ ] Quality of reasoning in outputs
- [ ] Limitations encountered
- [ ] Possible improvements

### 5. Code & Data
- [ ] This notebook (commented and runnable)
- [ ] Generated datasets (stage1/stage2)
- [ ] Model checkpoints or download links

---
## 🚀 Next Steps

1. **Run the notebook cells sequentially**
2. **Monitor API usage** (respect rate limits)
3. **Adjust hyperparameters** if needed
4. **Document your observations** in markdown cells
5. **Export results** for your report

Good luck! 🎓

In [ ]:
%cd /kaggle/working/LLaMA-Factory/

# Stage 2 training - LOAD Stage 1 adapter and continue training
!llamafactory-cli train \
    --stage sft \
    --do_train \
    --model_name_or_path unsloth/Qwen2.5-7B-Instruct-bnb-4bit \
    --adapter_name_or_path /kaggle/working/tp4_dasd/outputs/stage1 \
    --dataset dasd_stage2 \
    --template qwen3_nothink \
    --finetuning_type lora \
    --lora_rank 8 \
    --lora_target all \
    --output_dir /kaggle/working/tp4_dasd/outputs/stage2 \
    --overwrite_output_dir \
    --plot_loss \
    --trust_remote_code \
    --per_device_train_batch_size 2 \
    --gradient_accumulation_steps 4 \
    --learning_rate 3.0e-5 \
    --num_train_epochs 2.0 \
    --lr_scheduler_type cosine \
    --warmup_ratio 0.1 \
    --logging_steps 10 \
    --save_steps 100 \
    --cutoff_len 2048 \
    --preprocessing_num_workers 8 \
    --fp16 \
    --report_to none

print("\n✅ Stage 2 training completed!")